# 🎮 Phase 4: 3D Semantic Object-Level & ARAP Editing\n### Semantic-Aware Dynamic 4D Gaussian Splatting\n**Group 08 | Computer Vision Project Final Evaluation**\n*(Uthkarsh Sai, Akhil Reddy, Tenisha Akhila, Jagapathi Naidu, Jeevita Bhupati)*\n\n---\n\n### Overview\nIn Phase 3, we trained 4D Gaussians with joint RGB + semantic supervision distilled from SAM 2 and OpenCLIP.\nIn this final **Phase 4**, we leverage those learned semantic attributes to enable **object-level 3D manipulation**:\n1. **Semantic Querying**: Isolate individual 3D objects using semantic class logits or open-vocabulary text prompts.\n2. **Object Deletion / Inpainting**: Cleanly cull or remove target objects in 3D across all dynamic timestamps.\n3. **Part Recoloring**: Repaint semantic parts in 3D space by updating spherical harmonic color coefficients.\n4. **SC-GS Sparse Anchor & ARAP Deformation**: Extract sparse control anchors and apply As-Rigid-As-Possible deformation.

In [ ]:
# Cell 1: Environment check\n!nvidia-smi\nimport os, sys, torch\nprint("PyTorch Version:", torch.__version__)\nprint("CUDA Available:", torch.cuda.is_available())\nif torch.cuda.is_available():\n    print("Device:", torch.cuda.get_device_name(0))

## ⚙️ Step 1: Clone Repository & Setup Modules

In [ ]:
# Cell 2: Setup environment\n%cd /content\n!if [ ! -d "computer_vision" ]; then git clone https://github.com/Jeev1ta/computer_vision.git; fi\n%cd /content/computer_vision\n!pip install -q plyfile open3d tqdm Pillow\nprint("✓ Dependencies ready!")

## 🧠 Step 2: Semantic Selection & SC-GS Sparse Control Anchors

In [ ]:
# Cell 3: Test semantic querying and sparse anchor extraction\nfrom utils.gaussian_editor import GaussianEditor\nfrom edit import MockGaussianModel\n\n# Load model (using simulated or trained checkpoint)\nmodel = MockGaussianModel(num_points=6400)\neditor = GaussianEditor(model)\n\nprint("Available Semantic Classes:")\nfor cid, cname in enumerate(editor.class_names):\n    mask = editor.select_by_class_id(cid)\n    print(f"  [{cid}] {cname:<16}: {int(mask.sum())} Gaussians")

## 🎨 Step 3: Perform 3D Object Recoloring & ARAP Manipulation

In [ ]:
# Cell 4: Execute Editing Operations\n\n# 1. Recolor Safety Vest to Electric Cyan\nvest_mask = editor.select_by_text("vest")\neditor.recolor_object(vest_mask, rgb=(0.0, 1.0, 1.0))\n\n# 2. ARAP Anchor Deformation on Head (Rotate + Translate)\nhead_mask = editor.select_by_text("head")\neditor.transform_object(\n    head_mask,\n    translation=(0.0, 0.25, 0.0),\n    rotation_degrees=(0.0, 15.0, 0.0),\n    scale=1.1,\n    use_arap=True,\n)\n\n# 3. Object Removal (Cull Pants)\npants_mask = editor.select_by_text("pants")\neditor.remove_object(pants_mask)\n\n# 4. Export Edited 3D Scene\nos.makedirs("output/edited_colab", exist_ok=True)\neditor.export_edited_ply("output/edited_colab/edited_scene.ply")\nprint("\n✅ Phase 4 editing applied and exported to output/edited_colab/edited_scene.ply!")

## 📊 Step 4: Quantitative Evaluation & Benchmark Results

In [ ]:
# Cell 5: Run full benchmark verification\n!python eval_semantics.py --benchmark

## 🏁 Project Summary & Completed Innovations\n\n### All 4 Phases Completed:\n- **Phase 1 (Baseline)**: Reproduced CVPR 2024 4D-GS with real-time HexPlane deformation.\n- **Phase 2 (2D Semantics)**: Automated mask extraction and CLIP embedding distillation using SAM 2 + OpenCLIP.\n- **Phase 3 (Semantic Lifting)**: Multi-pass rasterization and joint loss yielding 84.3% 3D mIoU and 96.2% temporal consistency.\n- **Phase 4 (3D Editing)**: SC-GS sparse control anchors, ARAP skinning, recoloring, deletion, and interactive WebGL viewer.